In [1]:
from config_file import *
sys.path.append('./problems')
from train import *
import math
import pandas as pd

In [2]:
print_problem()

Multifeature Problem: OR(AND(circle,A), AND(!triangle, B)); TRAINING =  ft_none_none ; INPUT =  fusion ; OUT_SIZE =  1 ; WTS =  single ; TEMP_FREEZE =  False


# Important!

The T1 files are actually T2 and vice versa. In this file, I am using the computed 'T1'formulas for T2 .

# Functions to compute GP features

## Comp comp


In [3]:
def gp_cc_T1(tab_fts): # I am switching them here so keep note of 
    # copy expression here
    # [If((x_1 < x_2)) Then((x_1 * 0.156827)) Else((0.090343 < x_1))]
    tab_fts = np.asarray(tab_fts)
    y = []
    for i in range(tab_fts.shape[0]):
        x_1 = tab_fts[i,1]
        x_2 = tab_fts[i,2]
        
        if x_1 < x_2: r = x_1 * 0.156827

        else:
            r = int(0.090343 < x_1)
        
        if r > 1: y.append(1)
        elif r < 0: y.append(0)
        else: y.append(r)
        
    return y

def gp_cc_T2(tab_fts):
    # copy expression here
    # [If((x_3 > x_4)) Then([-](-0.902807)) Else((0.156936 * x_3))]
    tab_fts = np.asarray(tab_fts)
    y = []
    for i in range(tab_fts.shape[0]):
        x_3 = tab_fts[i,3]
        x_4 = tab_fts[i,4]
        
        if x_3 > x_4: r = 0.902807

        else:
            r = 0.156936 * x_3
        
        if r > 1: y.append(1)
        elif r < 0: y.append(0)
        else: y.append(r)
        
    return y

## None none

In [4]:
def gp_nn_T1(tab_fts): # I am switching them here so keep note of 
    # copy expression here
    # ((x_2 * 0.751749) / (x_1 + x_2))
    tab_fts = np.asarray(tab_fts)
    y = []
    for i in range(tab_fts.shape[0]):
        x_1 = tab_fts[i,1]
        x_2 = tab_fts[i,2]
        
        r = (x_2 * 0.751749) / (x_1 + x_2)
        
        if r > 1: y.append(1)
        elif r < 0: y.append(0)
        else: y.append(r)
        
    return y

def gp_nn_T2(tab_fts):
    # copy expression here
    # [If((x_4 > x_3)) Then([If(1) Then(0.655005) Else(x_3)]) Else(1/(3.198801))]
    tab_fts = np.asarray(tab_fts)
    y = []
    for i in range(tab_fts.shape[0]):
        x_3 = tab_fts[i,3]
        x_4 = tab_fts[i,4]
        
        if x_4 > x_3: r = 0.655005

        else:
            r = 0.312617
        
        if r > 1: y.append(1)
        elif r < 0: y.append(0)
        else: y.append(r)
        
    return y

## Part none

In [5]:
def gp_pn_T1(tab_fts): # I am switching them here so keep note of 
    # copy expression here
    # [If((x_1 < x_2)) Then((-0.077198 / -1.318057)) Else((-0.937803)**2)]
    tab_fts = np.asarray(tab_fts)
    y = []
    for i in range(tab_fts.shape[0]):
        x_1 = tab_fts[i,1]
        x_2 = tab_fts[i,2]
        
        if x_1 < x_2: r = 0.05857

        else:
            r = 0.879474
        
        if r > 1: y.append(1)
        elif r < 0: y.append(0)
        else: y.append(r)
        
    return y

def gp_pn_T2(tab_fts):
    # copy expression here
    # [If((x_4 > x_3)) Then(1/(1.163827)) Else((0.958297 / 3.951327))]
    tab_fts = np.asarray(tab_fts)
    y = []
    for i in range(tab_fts.shape[0]):
        x_3 = tab_fts[i,3]
        x_4 = tab_fts[i,4]
        
        if x_4 > x_3: r = 0.859234

        else:
            r = 0.242525
        
        if r > 1: y.append(1)
        elif r < 0: y.append(0)
        else: y.append(r)
        
    return y

## Part part

In [6]:
def gp_pp_T1(tab_fts): # I am switching them here so keep note of 
    # copy expression here
    # [If((x_1 > x_2)) Then((1.021008 + -0.068676)) Else((0.169314 * x_1))]
    tab_fts = np.asarray(tab_fts)
    y = []
    for i in range(tab_fts.shape[0]):
        x_1 = tab_fts[i,1]
        x_2 = tab_fts[i,2]
        
        if x_1 > x_2: r = 0.952332

        else:
            r = 0.169314 * x_1
        
        if r > 1: y.append(1)
        elif r < 0: y.append(0)
        else: y.append(r)
        
    return y

def gp_pp_T2(tab_fts):
    # copy expression here
    # [If((x_3 < x_4)) Then((x_3 / 2.853253)) Else([If(0) Then(0) Else(0.648319)])]
    tab_fts = np.asarray(tab_fts)
    y = []
    for i in range(tab_fts.shape[0]):
        x_3 = tab_fts[i,3]
        x_4 = tab_fts[i,4]
        
        if x_3 < x_4: r = x_3 / 2.853253

        else:
            r = 0.648319
        
        if r > 1: y.append(1)
        elif r < 0: y.append(0)
        else: y.append(r)
        
    return y

# Using expressions to replace features in csv files

In [7]:
def make_gp_fts_df(file_dir, t1_func, t2_func):
    train_data = pd.read_csv(file_dir + f'train_dl_I_T_Y.csv')
    test_data = pd.read_csv(file_dir + f'test_dl_I_T_Y.csv')
    train_fts = pd.read_csv(file_dir + f'train_tab_T.csv')
    test_fts = pd.read_csv(file_dir + f'test_tab_T.csv')
    
    # calculate new T ft(s) 
    train_T1 = t1_func(train_fts)
    test_T1 = t1_func(test_fts)
    train_T2 = t2_func(train_fts)
    test_T2 = t2_func(test_fts)

    # replace T in fusion file
    train_data['T1'] = train_T1
    test_data['T1'] = test_T1
    train_data['T2'] = train_T2
    test_data['T2'] = test_T2

    # save new files
    train_file = file_dir + f'train_I_T_Y.csv'
    train_data.to_csv(train_file, index=False)
    print(f"Saved {train_file}!")

    test_file = file_dir + f'test_I_T_Y.csv'
    test_data.to_csv(test_file, index=False)
    print(f"Saved {test_file}!")

In [8]:
# model specific dirs
comp_comp_dir = '/export/scratch2/ima/MultiFIX_GECCO25_code/scripts/gp_files/Multifeature/ft_comp_comp_single_False/'
none_none_dir = '/export/scratch2/ima/MultiFIX_GECCO25_code/scripts/gp_files/Multifeature/ft_none_none_single_False/'
part_part_dir = '/export/scratch2/ima/MultiFIX_GECCO25_code/scripts/gp_files/Multifeature/ft_part_part_single_False/'
part_none_dir = '/export/scratch2/ima/MultiFIX_GECCO25_code/scripts/gp_files/Multifeature/ft_part_none_single_False/'

# using function to save new files
save = False
if save:
    make_gp_fts_df(comp_comp_dir, gp_cc_T1, gp_cc_T2)
    make_gp_fts_df(none_none_dir, gp_nn_T1, gp_nn_T2)
    make_gp_fts_df(part_part_dir, gp_pp_T1, gp_pp_T2)
    make_gp_fts_df(part_none_dir, gp_pn_T1, gp_pn_T2)

## Calculating performance for GP fusion with GP tab

In [9]:
# GP symbolic expression for prediction
def gp_cc_Y(tab_fts):
    # copy expression here
    # [If((x_3 < x_1)) Then((x_0 * x_2)) Else((0.207136 + x_3))]
    tab_fts = np.asarray(tab_fts)
    y = []
    for i in range(tab_fts.shape[0]):
        x_0 = tab_fts[i,0]
        x_1 = tab_fts[i,1]
        x_2 = tab_fts[i,2]
        x_3 = tab_fts[i,3]

        if x_3 < x_1: y_i = x_0 * x_2

        else: y_i = 0.207136 + x_3
        
        y.append(y_i)
        
    return y

# GP symbolic expression for prediction
def gp_nn_Y(tab_fts):
    # copy expression here
    # [If((0.498193 > x_0)) Then((x_1 - x_2)) Else((x_1 > x_3))]
    tab_fts = np.asarray(tab_fts)
    y = []
    for i in range(tab_fts.shape[0]):
        x_0 = tab_fts[i,0]
        x_1 = tab_fts[i,1]
        x_2 = tab_fts[i,2]
        x_3 = tab_fts[i,3]
        
        if 0.498193 > x_0: y_i = x_1 - x_2

        else: y_i = x_1 > x_3

        y.append(y_i)

    return y

# GP symbolic expression for prediction
def gp_pn_Y(tab_fts):
    # copy expression here
    # [If((x_1 > x_3)) Then((-0.991255)**2) Else((x_2 * x_0))]
    tab_fts = np.asarray(tab_fts)
    y = []
    for i in range(tab_fts.shape[0]):
        x_0 = tab_fts[i,0]
        x_1 = tab_fts[i,1]
        x_2 = tab_fts[i,2]
        x_3 = tab_fts[i,3]
        
        if x_1 > x_3: y_i = 0.982586

        else: y_i = x_2 * x_0

        y.append(y_i)

    return y

# GP symbolic expression for prediction
def gp_pp_Y(tab_fts):
    # copy expression here
    # [If((x_3 > x_1)) Then((0.585634 < x_3)) Else((x_0 * x_2))] ← using this one
    # [If((x_3 < x_1)) Then((x_2 * x_0)) Else((x_3 > 0.496917))]
    tab_fts = np.asarray(tab_fts)
    y = []
    for i in range(tab_fts.shape[0]):
        x_0 = tab_fts[i,0]
        x_1 = tab_fts[i,1]
        x_2 = tab_fts[i,2]
        x_3 = tab_fts[i,3]

        if x_3 > x_1: y_i = 0.585634 < x_3

        else: y_i = x_0 * x_2

        y.append(y_i)

    return y

In [14]:
# function to calculate and print AUC and BAcc

def print_bacc(training):
    training_dict = {'comp_comp': [comp_comp_dir, gp_cc_Y],
                     'none_none': [none_none_dir, gp_nn_Y],
                     'part_part': [part_part_dir, gp_pp_Y], 
                     'part_none': [part_none_dir, gp_pn_Y]}
    
    gp_data = pd.read_csv(training_dict[training][0] + f'test_I_T_Y.csv')
    gt = np.asarray(gp_data['Y']).reshape(200,)
    gp = np.asarray(training_dict[training][1](gp_data)).reshape(200,)
    gp[gp<0.5] = 0
    gp[gp>=0.5] = 1

    bacc = balanced_accuracy_score(gt, gp)

    print(training)
    print("Final BAcc = ", round(bacc, 5), '\n')

In [15]:
print_bacc('comp_comp')
print_bacc('none_none')
print_bacc('part_part')
print_bacc('part_none')

comp_comp
Final BAcc =  0.93872 

none_none
Final BAcc =  0.81567 

part_part
Final BAcc =  0.92411 

part_none
Final BAcc =  0.89209 



# Visualisation

In [ ]:
# cc
